# Week 6: Text Clustering and Topic Modeling

In Week 5 we pushed a review through a stack of transformer encoder blocks, collapsed the token vectors into one vector with masked mean pooling, and handed that vector to a classification head trained on labels.

This week we cut off the head and throw away the labels. We keep only the pooled vector, one per document, and ask a different question: if nobody tells the model what the categories are, can the geometry of those vectors reveal them on its own?

That is **text clustering**. When we go one step further and describe each cluster with a handful of words, it becomes **topic modeling**.
> **Setup (Colab or a fresh environment):** `pip install sentence-transformers umap-learn hdbscan bertopic`

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import re, html, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
import transformers
transformers.logging.disable_progress_bar()

np.random.seed(0)
_ = torch.manual_seed(0)

---
# Text Clustering

## 1. Embedding documents

### 1.1 The running example

Every toy example in this notebook uses the same ten headlines: three about sports, three about finance, three about technology, and one that belongs to none of them.

| id | headline | (hidden) topic |
|---|---|---|
| s1 | the striker scored the winning goal | sports |
| s2 | our team conceded one goal but won | sports |
| s3 | the forward netted his late winner | sports |
| f1 | the market fell as shares dropped | finance |
| f2 | the stock price rose after strong earnings | finance |
| f3 | investors sold shares in the market | finance |
| t1 | the new phone has the fastest chip | tech |
| t2 | the company released new mobile software | tech |
| t3 | the chip maker unveiled its processor | tech |
| o  | grandma baked an apple pie | none |

Look closely at **s1** and **s3**. They describe the same event, yet they share no content word: striker/forward, scored/netted, winning goal/late winner. The only word they have in common is "the". That pair is going to break the first thing we try.

In [ ]:
headlines = {
    "s1": "the striker scored the winning goal",
    "s2": "our team conceded one goal but won",
    "s3": "the forward netted his late winner",
    "f1": "the market fell as shares dropped",
    "f2": "the stock price rose after strong earnings",
    "f3": "investors sold shares in the market",
    "t1": "the new phone has the fastest chip",
    "t2": "the company released new mobile software",
    "t3": "the chip maker unveiled its processor",
    "o":  "grandma baked an apple pie",
}
ids = list(headlines)
texts = list(headlines.values())

### 1.2 Naive attempt: bag-of-words vectors

The simplest way to turn a sentence into a vector is to count words. Build a vocabulary, give each word one dimension, and store how many times each word appears. To compare two vectors, we use cosine similarity, the same measure used for nearest neighbours in Week 2:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\lVert \mathbf{u} \rVert \, \lVert \mathbf{v} \rVert}$$

For word-count vectors it ranges from 0 (no words in common) to 1 (identical word proportions). Below, we compare s1 against its paraphrase s3, and against the unrelated finance headline f1.

In [ ]:
cv = CountVectorizer()
bow = cv.fit_transform([headlines["s1"], headlines["s3"], headlines["f1"]]).toarray()
print(pd.DataFrame(bow, index=["s1", "s3", "f1"], columns=cv.get_feature_names_out()).to_string())

def cosine(u, v):
    return u @ v / (np.linalg.norm(u) * np.linalg.norm(v))

print(f"\nbag-of-words cos(s1, s3) = {cosine(bow[0], bow[1]):.3f}")
print(f"bag-of-words cos(s1, f1) = {cosine(bow[0], bow[2]):.3f}")

A perfect tie. Look at the count table: the only column where s1 overlaps with s3, or with f1, is "the". Word counting thinks a stock-market headline is exactly as close to s1 as s1's own paraphrase, and the one word driving both scores carries no meaning at all. Removing stop words does not rescue it, it just turns both scores into 0.

### 1.3 The fix: a pretrained sentence embedding model

What we need is a vector that captures meaning, so "striker" and "forward" land near each other even though they are different strings. We already know how to build one: Week 5's encoder turns a sentence into a single pooled vector. The problem in Week 5 was that our encoder saw only 3,200 labelled reviews and never learned much about language.

So we borrow one that has. `all-MiniLM-L6-v2` is a 6-layer transformer encoder (the same bidirectional block shape as Week 5's `TransformerEncoderBlock`), producing 384-dimensional token vectors, which are then combined into one vector per sentence using exactly Week 5's masked mean pooling. A final step scales every vector to length 1, so cosine similarity becomes a plain dot product. It was trained on over a billion sentence pairs so that sentences with similar meaning end up close together. 

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = model.encode(texts)
print("embedding matrix shape:", emb.shape)

i = {k: n for n, k in enumerate(ids)}
print(f"\nembedding cos(s1, s3) = {cosine(emb[i['s1']], emb[i['s3']]):.3f}")
print(f"embedding cos(s1, f1) = {cosine(emb[i['s1']], emb[i['f1']]):.3f}")

The tie is broken decisively: 0.380 for the paraphrase versus 0.043 for the finance headline, even though the two sports headlines share no content word. The model has learned that "netted his late winner" and "scored the winning goal" describe the same kind of event.

Here is the full picture for all ten headlines. Three bright blocks along the diagonal, one per topic, and one row (the pie) that is dim everywhere. Grouping those blocks automatically, and setting the pie aside, is exactly what clustering has to do.

## 2. Real data: AG News

AG News is a collection of news articles from over 2,000 sources, each tagged as World, Sports, Business, or Sci/Tech. We will not use the labels to cluster. They stay locked away until Section 4, where we use them only to check what the clusters found.

In [ ]:
url = "https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv"
df = pd.read_csv(url, header=None, names=["label", "title", "desc"])
label_names = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

def clean(s):
    s = s.replace("\\", " ")
    s = re.sub(r"#(\d+);", lambda m: chr(int(m.group(1))), s)   # "#39;" -> "'"
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)                              # stray HTML tags
    return re.sub(r"\s+", " ", s).strip()

df["text"] = (df.title + ". " + df.desc).map(clean)
news = (df.groupby("label").sample(500, random_state=0)
          .sample(frac=1, random_state=0).reset_index(drop=True))
docs = news.text.tolist()
hidden_labels = news.label.map(label_names).values     # locked away until Section 4

print(len(df), "articles available,", len(docs), "sampled")
for d in docs[:3]:
    print("-", d[:110], "...")

In [ ]:
embeddings = model.encode(docs, batch_size=64, show_progress_bar=False)
print("embeddings:", embeddings.shape)

## 3. Clustering the embeddings

**What we are looking for.** The goal of clustering is *not* to reproduce the four labels. It is to find groups of articles that are about the same thing, without being told what the groups are. A good result places most articles into some group, and each group is about one subject. 

### 3.1 The clustering algorithm: HDBSCAN

The algorithm we use is **HDBSCAN**, a density-based method. In plain terms, it looks for crowded regions of the space and calls each crowded region a cluster. Documents that sit in empty space between crowded regions are labelled **noise**, with cluster id $-1$.

If you have met k-means before, two differences matter here. With HDBSCAN you do not tell it how many clusters to find, which is good, because in a real corpus you do not know how many topics there are. And it does not force every document into a cluster, so a document that fits nowhere is set aside instead of distorting a cluster it does not belong to.

Two settings control it. `min_cluster_size` is the smallest group you are willing to call a cluster; we use 20 articles. `min_samples` is how many neighbours it looks at when judging how crowded a region is; we use 5. Larger values give fewer, broader clusters; smaller values give more, finer ones.

### 3.2 Naive attempt: cluster the 384-dimensional vectors directly

The obvious move is to run HDBSCAN on the 384-dimensional embeddings as they are.

In [ ]:
def summary(name, labels):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    placed = np.mean(labels != -1)
    print(f"{name:<24} clusters = {n_clusters:>3}   articles placed in a cluster = {placed:6.1%}")

raw_labels = HDBSCAN(min_cluster_size=20, min_samples=5).fit(embeddings).labels_
summary("HDBSCAN on raw 384-d", raw_labels)

HDBSCAN placed only a third of the articles into clusters and labelled the other two thirds as noise. It could only find the very easiest, most tightly packed groups.

**Why:** in 384 dimensions, distances stop being informative. Every article ends up roughly the same distance from every other article, so there are no regions that look clearly more crowded than the rest. HDBSCAN depends entirely on spotting crowded regions, so it finds very few and gives up on most of the data. This is known as the **curse of dimensionality**.

### 3.3 The fix: UMAP

Real embeddings are not spread evenly through all 384 dimensions. Articles about baseball occupy a small, structured region of the space. **UMAP** (Uniform Manifold Approximation and Projection) squeezes the data down to a handful of dimensions while keeping that structure. In two steps:

1. In the original space, find each article's nearest neighbours and build a graph connecting them.
2. Place the articles in a low-dimensional space (here 5 dimensions) and move them around until the same articles are neighbours there.

UMAP cares about *who is next to whom*, which is exactly what HDBSCAN needs. The settings follow the textbook: 5 output dimensions, `min_dist=0.0` so articles in the same neighbourhood are allowed to pack tightly, and cosine as the input metric to match how the embeddings were trained. `random_state` makes the run reproducible.

We run the same HDBSCAN, with the same settings, on the UMAP output.

In [ ]:
umap_model = UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42)
reduced = umap_model.fit_transform(embeddings)
labels = HDBSCAN(min_cluster_size=20, min_samples=5).fit(reduced).labels_

summary("HDBSCAN on raw 384-d", raw_labels)
summary("UMAP 5-d, then HDBSCAN", labels)

Same articles, same clustering algorithm, same settings. With UMAP in front of it, HDBSCAN places 70% of the articles instead of 33%. The next question is whether each of those clusters is really about one subject.

## 4. Checking clusters against the hidden labels

In a real project there are no labels, and the only check is reading articles from each cluster. Here we are lucky, so we can measure. The simplest measure is **purity**: for each cluster, count how many of its articles share the cluster's most common label, add those counts up, and divide by the number of clustered articles (noise excluded). A purity of 1.0 means every cluster contains only one category.

In [ ]:
def purity(cluster_labels, true_labels):
    keep = cluster_labels != -1
    total = 0
    for c in set(cluster_labels[keep]):
        total += collections.Counter(true_labels[cluster_labels == c]).most_common(1)[0][1]
    return total / keep.sum()

print(f"purity of the UMAP + HDBSCAN clusters = {purity(labels, hidden_labels):.3f}")

More than 90% of the clustered articles share their cluster's majority label, so the clusters really are about one subject each. 

The plot below shows all of this at once. Both panels draw **the same 2-D UMAP view of the same 2,000 articles** (a separate 2-D run, used only for the picture); only the colouring differs. On the right, each article is coloured by its hidden label. On the left, each article is coloured by its cluster, and every cluster is painted in the colour of its majority label, so you can see which label region each cluster lives in.

In [ ]:
coords_2d = UMAP(n_components=2, min_dist=0.0, metric="cosine", random_state=42).fit_transform(embeddings)
label_colour = {"World": "tab:blue", "Sports": "tab:green", "Business": "tab:orange", "Sci/Tech": "tab:purple"}
majority = {c: collections.Counter(hidden_labels[labels == c]).most_common(1)[0][0]
            for c in set(labels) if c != -1}
cluster_colour = ["lightgrey" if c == -1 else label_colour[majority[c]] for c in labels]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
axes[0].scatter(*coords_2d.T, s=5, c=cluster_colour)
axes[0].set_title("Coloured by cluster (found without labels)\neach cluster in the colour of its majority label, noise in grey")
axes[1].scatter(*coords_2d.T, s=5, c=[label_colour[l] for l in hidden_labels])
axes[1].set_title("Coloured by hidden label")
handles = [plt.Line2D([], [], marker="o", ls="", color=col, label=name) for name, col in label_colour.items()]
handles.append(plt.Line2D([], [], marker="o", ls="", color="lightgrey", label="noise (-1)"))
fig.legend(handles=handles, loc="lower center", ncol=5)
fig.suptitle("Same 2-D UMAP view of 2,000 articles, two colourings")
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(rect=(0, 0.06, 1, 1)); plt.show()

The two panels match closely: the clusters fall inside the label regions, which is what 90% purity looks like. The grey points on the left are the noise, mostly sitting on the borders between regions, where articles mix subjects. 



---
# Topic Modeling

## 5. From clusters to topics

To call a cluster a *topic*, we need a short description, usually a handful of words. The question is which words.

### 5.1 Naive attempt: most frequent words per cluster

Glue all documents in a cluster into one long document, count words, take the top few. We start with the toy headlines, grouped by their topic the way a good clustering would group them:

In [ ]:
toy_clusters = {
    "sports":  " ".join(texts[0:3]),
    "finance": " ".join(texts[3:6]),
    "tech":    " ".join(texts[6:9]),
}
cv = CountVectorizer()
X_toy = cv.fit_transform(toy_clusters.values()).toarray().astype(float)
vocab = cv.get_feature_names_out()

for name, row in zip(toy_clusters, X_toy):
    top = np.argsort(-row)[:3]
    print(f"{name:<8} most frequent:", [(vocab[j], int(row[j])) for j in top])

"the" wins every cluster. Raw frequency tells you what is common in English, not what is distinctive about a cluster.

### 5.2 The fix: c-TF-IDF

Classic TF-IDF, from information retrieval, scores a word highly when it is frequent in *one document* but rare *across documents*. c-TF-IDF (class-based TF-IDF) applies the same idea with **each cluster treated as one big document**. For a word $t$ in cluster $c$:

$$W_{t,c} = \underbrace{\frac{\text{tf}_{t,c}}{\sum_{t'} \text{tf}_{t',c}}}_{\text{how common in this cluster}} \times \underbrace{\log\left(1 + \frac{A}{f_t}\right)}_{\text{how rare overall}}$$

where $\text{tf}_{t,c}$ is how often $t$ appears in cluster $c$, $f_t$ is how often $t$ appears across all clusters, and $A$ is the average number of words per cluster.

Each toy cluster happens to contain exactly 19 words, so $A = 19$. Two words to compare:

| word | sports | finance | tech | $f_t$ |
|---|---|---|---|---|
| the | 3 | 3 | 4 | 10 |
| goal | 2 | 0 | 0 | 2 |

The rarity factors (natural log):

$$\text{the: } \log\left(1 + \tfrac{19}{10}\right) = \log(2.9) = 1.065 \qquad \text{goal: } \log\left(1 + \tfrac{19}{2}\right) = \log(10.5) = 2.351$$

The scores in the sports cluster:

$$W_{\text{the},\,\text{sports}} = \frac{3}{19} \times 1.065 = 0.158 \times 1.065 = 0.168$$

$$W_{\text{goal},\,\text{sports}} = \frac{2}{19} \times 2.351 = 0.105 \times 2.351 = 0.248$$

"the" appears more often in the sports cluster than "goal" does, but it appears everywhere, so its rarity factor is less than half of goal's. "goal" now wins.

In [ ]:
def c_tf_idf(X):
    '''X: (n_clusters, vocab) word counts, one row per cluster.'''
    tf = X / X.sum(axis=1, keepdims=True)          # how common in this cluster
    A = int(X.sum(axis=1).mean())                   # average words per cluster
    f = X.sum(axis=0)                               # frequency across all clusters
    return tf * np.log(1 + A / f)

W_toy = c_tf_idf(X_toy)
the, goal = list(vocab).index("the"), list(vocab).index("goal")
print(f"W(the, sports)  = {W_toy[0, the]:.4f}")
print(f"W(goal, sports) = {W_toy[0, goal]:.4f}\n")
for name, row in zip(toy_clusters, W_toy):
    top = np.argsort(-row)[:3]
    print(f"{name:<8} c-TF-IDF:", [(vocab[j], round(float(row[j]), 3)) for j in top])

Every cluster is now described by its own vocabulary: goal for sports, shares and market for finance, chip and new for tech.

Now the real clusters from previous class. We build the same counts for all clusters plus the noise group, and compare raw frequency against c-TF-IDF for the five largest.

In [ ]:
cluster_ids = sorted(set(labels))                   # includes -1 (noise), so its words still count toward rarity
joined = [" ".join(d for d, l in zip(docs, labels) if l == c) for c in cluster_ids]
cv_news = CountVectorizer()
X_news = cv_news.fit_transform(joined).toarray().astype(float)
vocab_news = cv_news.get_feature_names_out()
W_news = c_tf_idf(X_news)

largest = [c for c, _ in collections.Counter(labels).most_common() if c != -1][:5]
for c in largest:
    r = cluster_ids.index(c)
    print(f"cluster {c:>2} ({(labels == c).sum()} docs)")
    print("   most frequent:", list(vocab_news[np.argsort(-X_news[r])[:5]]))
    print("   c-TF-IDF:     ", list(vocab_news[np.argsort(-W_news[r])[:5]]))

### 5.4 Tagging every article with its topic

The last step is the actual output of topic modeling. Every article takes the name of the cluster it belongs to; articles HDBSCAN labelled as noise get "no topic".

In [ ]:
# Recount without stop words, then name each cluster after its top four c-TF-IDF words
cv_topics = CountVectorizer(stop_words="english")
W_topics = c_tf_idf(cv_topics.fit_transform(joined).toarray().astype(float))
vocab_topics = cv_topics.get_feature_names_out()

topic_names = {c: "_".join(vocab_topics[np.argsort(-W_topics[r])[:4]])
               for r, c in enumerate(cluster_ids) if c != -1}

# Every article takes its cluster's name; noise gets "no topic"
news["topic"] = [topic_names.get(c, "no topic") for c in labels]

sample = news.sample(8, random_state=4)
print(sample.assign(headline=sample.title.map(clean))[["headline", "topic"]].to_string(index=False))
print(f"\narticles with a topic: {np.mean(news.topic != 'no topic'):.1%}")

That is a complete topic model, built entirely from parts we understand: sentence embeddings, UMAP, HDBSCAN, and c-TF-IDF. Each of the 2,000 articles now carries a readable topic, or "no topic" when it fits nowhere.

## 6. Where this leaves us

The pipeline works, and every stage was introduced by first watching the naive version fail: word counts could not see meaning, raw 384-dimensional vectors could not be clustered, and raw frequencies could not describe a cluster.

It also has real limits worth keeping in view:

- **Almost a third of the articles get no topic.** HDBSCAN is honest about articles it cannot place, but that is still a lot of unexplained data.
- **One topic per article.** An article about a tech company's earnings belongs to both business and technology; clustering has to pick one.
- **The result depends on your settings.** Changing `min_cluster_size` or `min_samples` changes how many topics you get and how broad they are.
- **"Similar" means whatever the embedding model learned it means.** Our model was trained to place sentences by subject, so our topics are subjects. A different embedding model would organise the same articles differently.

### Why does a library exist for this?

The four stages we built are exactly what the **BERTopic** library packages. For the 2,000 articles we already have, we do not need it. Our version starts to struggle in three situations:

- **A new article arrives.** Our pipeline only tags the articles it was built on. Placing a new one means keeping the fitted UMAP, projecting the article with it, and asking HDBSCAN which cluster it falls into. BERTopic does all of that in one call.
- **You want to swap a stage.** Trying a different embedding model, reducer, or clustering algorithm means rewiring our code. In BERTopic each stage is one argument.
- **You want better names.** BERTopic lets you plug in smarter ways of choosing topic words, including asking a generative model for a label.

The exercise explores all three.

## Exercise: rebuilding the pipeline with BERTopic

Do this after Class 2. Run all cells above first, so `docs`, `embeddings`, `labels`, `model`, and `topic_table` exist.

### Part A: Reproduce our topic model

Create a `BERTopic` model that uses **our exact stages**:

- `embedding_model=model`
- `umap_model=UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42)`
- `hdbscan_model=HDBSCAN(min_cluster_size=20, min_samples=5, prediction_data=True)` (the last argument lets HDBSCAN place new articles later, which Part B needs)
- `vectorizer_model=CountVectorizer(stop_words="english")`

Fit it with `topic_model.fit_transform(docs, embeddings)`. Passing our `embeddings` means BERTopic does not re-embed the articles.

1. Use `adjusted_rand_score(labels, topics)` to compare BERTopic's grouping with ours. This score compares two groupings while ignoring what the groups are called; 1.0 means identical. What do you get?
2. Display `topic_model.get_topic_info()` and compare the names with `topic_table`. The words should match, but the topic numbers do not. Why?

In [ ]:
from bertopic import BERTopic

# TODO: build topic_model with the four stages listed above
# TODO: topics, _ = topic_model.fit_transform(docs, embeddings)
# TODO: print adjusted_rand_score(labels, topics) and display topic_model.get_topic_info().head(10)

### Part B: Topics for articles the model has never seen

Use `topic_model.transform(new_headlines)` to assign topics to the headlines below, then print each headline next to its topic name (the `Name` column of `get_topic_info()`).

1. Which headlines landed in a sensible topic? Which did not, and what topic id did they get?
2. Explain in your own words why the last three headlines were treated the way they were.
3. Using Section 6, describe what our manual pipeline would need in order to do the same thing.

In [ ]:
new_headlines = [
    "Red Sox rally past the Yankees in the ninth inning",
    "Microsoft patches a security hole in Windows XP",
    "Oil prices climb as crude supplies tighten",
    "NASA scientists study samples from the Genesis capsule",
    "Grandma's secret recipe for the perfect apple pie",
    "Ten tips for planting tulip bulbs this autumn",
    "Pop star announces divorce after two years of marriage",
]

# TODO: new_topics, _ = topic_model.transform(new_headlines)
# TODO: print each headline with its topic id and name

### Part C: Better topic names

c-TF-IDF chooses words purely by counting. `KeyBERTInspired` takes c-TF-IDF's candidate words and re-ranks them by how close each word's embedding is to the embeddings of the topic's most representative articles, so the same embedding model that grouped the articles now helps name them.

Update the topic words without reclustering:

```python
from bertopic.representation import KeyBERTInspired
topic_model.update_topics(docs, vectorizer_model=CountVectorizer(stop_words="english"),
                          representation_model=KeyBERTInspired())
```

1. Display the names of the ten largest topics before and after the update.
2. Pick two topics whose name improved and one that did not, and explain why in a sentence each.
3. What happened to "reuters"?

In [ ]:
# TODO: save the current names, run update_topics with KeyBERTInspired, compare before and after